# 03 · Naive OLS

Estimate the raw association between cabin categories and log revenue per berth. Document the upward bias. This is the 'before' in the before/after identification story.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel  = pd.read_parquet(DATA_DIR / "ship_month_panel.parquet")
events = pd.read_csv(DATA_DIR / "category_change_events.csv", parse_dates=["event_month"])

print(f"Panel: {panel.shape}  |  Events: {len(events)}")

In [ ]:
import statsmodels.formula.api as smf

## Model 1 · Bivariate (no controls)

In [ ]:
m1 = smf.ols("log_revenue_per_berth ~ cabin_category_count", data=panel).fit(cov_type="HC3")
print(m1.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|", "[0.025", "0.975]"]])
print(f"\nβ_naive = {m1.params['cabin_category_count']:.4f}  (true β = 0.06)")
print(f"Overstatement: {m1.params['cabin_category_count']/0.06:.1f}×")

## Model 2 · With observable controls (partial adjustment)

In [ ]:
m2 = smf.ols(
    "log_revenue_per_berth ~ cabin_category_count + np.log(berth_capacity)"
    " + C(brand_tier) + C(sailing_region)",
    data=panel
).fit(cov_type="HC3")

coef = m2.params["cabin_category_count"]
ci   = m2.conf_int().loc["cabin_category_count"]
print(f"β_controlled = {coef:.4f}  95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"True β       = 0.06")
print(f"Remaining bias: {(coef - 0.06)/0.06*100:.0f}%")
print()
print("Controls reduce bias but don't eliminate it — unobserved scale still confounds.")

## Coefficient comparison table

In [ ]:
results = {
    "Bivariate OLS":    m1.params["cabin_category_count"],
    "OLS + Controls":   m2.params["cabin_category_count"],
    "True β (DGP)":     0.06,
}
for name, coef in results.items():
    bar = "█" * int(coef * 100)
    print(f"{name:<22}  {coef:.4f}  {bar}")

print()
print("Panel FE and event study (notebooks 04, 08) will bring this toward 0.06.")

## Why OLS fails: DAG intuition

In [ ]:
print("""
DAG:

  Ship Scale ──→ Cabin Category Count ──→ log Revenue per Berth
       │                                           ↑
       └───────────────────────────────────────────┘
                  (via ship fixed effect)

Ship Scale → Cabin Category Count: larger, more sophisticated ships have more categories.
Ship Scale → log Revenue per Berth: larger ships also command higher ADR / yield.

OLS estimate = causal effect + confounding bias from the backdoor path through Ship Scale.
Observable proxies (brand_tier, berth_capacity) are partial: they capture some of the
scale variation but not all. The residual unobserved scale is the OVB.

Fix: ship fixed effects block the backdoor path entirely (notebook 04).
""")